# Multilingual Health QA — Low-Resource African Languages
**Zindi Competition | Machine Learning Techniques I — Final Project**

| | |
|---|---|
| **Dataset** | 29,815 training records · 8 subsets (`ID`, `input`, `output`, `subset`) |
| **Task** | Seq2seq health QA — ROUGE-1 F1, ROUGE-L F1, LLM-as-a-Judge |
| **Package** | All reusable logic lives in `src/mhqa/` — notebook calls clean functions |
| **Key EDA decisions** | `MAX_INPUT=128` (100% fit) · `MAX_TARGET=384` (99%+ fit) · subset-prefixed prompt · 467 intra-subset duplicates dropped |

**Run order:** 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11


## 0. Environment Setup

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# In VS Code terminal:   pip install -r requirements.txt
# In Colab (uncomment):
# !pip install -q transformers==4.40.0 datasets peft accelerate bitsandbytes \
#               rouge_score evaluate sentencepiece sacrebleu scikit-learn \
#               pandas numpy matplotlib seaborn tqdm pyyaml ipywidgets


In [ ]:
import os, sys, json, random, warnings
from pathlib import Path

# ── Add src/ to path so mhqa package is importable ───────────────────────────
PROJECT_ROOT = Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
)

# ── mhqa package imports ──────────────────────────────────────────────────────
from mhqa.constants   import ID_COL, QUESTION_COL, ANSWER_COL, SUBSET_COL, SUBSET_ORDER, SUBSET_LABELS
from mhqa.config      import TrainingConfig, load_config, select_config
from mhqa.data        import load_all, load_split, stratified_split, make_hf_datasets
from mhqa.modeling    import load_tokenizer, load_model
from mhqa.metrics     import compute_rouge, compute_rouge_by_language, make_compute_metrics
from mhqa.retrieval   import PerLanguageRetriever
from mhqa.infer       import generate_batch, predict_dataframe
from mhqa.evaluate    import holdout_for, evaluate_model
from mhqa.submit      import make_submission
from mhqa.eda         import (plot_eda_overview, plot_token_budget,
                               plot_per_subset_rouge, plot_training_curves,
                               plot_leaderboard_progression)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")
    print("Tip    : use Google Colab (Runtime → Change runtime type → GPU T4)")

print("\nAll mhqa imports successful.")


## 1. Data Loading

In [ ]:
# ── Path config — works identically in VS Code (local) and Colab ─────────────
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR   = Path("/content/drive/MyDrive/multilingual_health_qa")
    OUTPUT_DIR = Path("/content/outputs")
    # Symlink src so mhqa is importable in Colab
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/kelvintawe12/MultiHealthQA-test.git",
                    "/content/repo"], check=False)
else:
    DATA_DIR   = Path("data")       # local: data/ folder in project root
    OUTPUT_DIR = Path("outputs")

for d in [OUTPUT_DIR / "plots", OUTPUT_DIR / "submissions",
          OUTPUT_DIR / "checkpoints", OUTPUT_DIR / "best_model"]:
    d.mkdir(parents=True, exist_ok=True)

# ── Load and clean all splits ─────────────────────────────────────────────────
print("Loading data...")
data     = load_all(DATA_DIR)           # cleans + deduplicates train, prompts all splits
train_df = data["train"]
val_df   = data["val"]
test_df  = data["test"]
sample   = pd.read_csv(DATA_DIR / "SampleSubmission.csv")

print(f"\nTrain : {train_df.shape}")
print(f"Val   : {val_df.shape}")
print(f"Test  : {test_df.shape}")
train_df.head(3)


## 2. Exploratory Data Analysis

EDA findings that directly determine every modelling decision below:

| Finding | Decision |
|---|---|
| All 29,815 inputs fit within 90 words (~128 mT5 tokens) | `MAX_INPUT_LEN = 128` |
| Akan answers p95 = 208 words → 256-token budget only covers 67.8% of all answers | `MAX_TARGET_LEN = 384` (99%+ overall) |
| 8 distinct subsets — English appears in 4 country configs | Use full `subset` tag in prompt, not just language name |
| 1,469 duplicate inputs: 1,002 cross-subset, 467 intra-subset | Drop intra-subset dups; retain cross-subset |
| 1 empty input (`ID_TR_Eng_Uga_E9A002A4`) | Dropped in `load_all()` |
| Amharic answers very short (median 19w) vs Akan (median 100w) | Cosine LR schedule; label smoothing = 0.1 |


In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
train_df["q_words"] = train_df[QUESTION_COL].str.split().str.len()
train_df["a_words"] = train_df[ANSWER_COL].str.split().str.len()

print("=== Subset distribution ===")
dist = train_df[SUBSET_COL].value_counts().reindex(SUBSET_ORDER)
for s, n in dist.items():
    print(f"  {SUBSET_LABELS[s]:<25} {n:>6,}  ({n/len(train_df)*100:.1f}%)")

print("\n=== Answer length per subset (words) ===")
stats = train_df.groupby(SUBSET_COL)["a_words"].agg(["median","mean","max"]).round(1)
stats.index = [SUBSET_LABELS[s] for s in stats.index]
print(stats.to_string())


In [ ]:
# ── Figure 1: EDA Overview ────────────────────────────────────────────────────
plot_eda_overview(train_df, OUTPUT_DIR / "plots")


In [ ]:
# ── Figure 2: Token Budget Analysis ──────────────────────────────────────────
plot_token_budget(train_df, OUTPUT_DIR / "plots")


## 3. Retrieval Baseline

A TF-IDF nearest-neighbour retriever is trained per subset (language-country config)
to establish a non-neural lower bound. Retrieved answers from EXP-01 justify the need
for fine-tuned seq2seq: the retriever produces grammatically valid answers but lacks
generative flexibility for paraphrase and cross-lingual adaptation.


In [ ]:
# Stratified 5% holdout from train for the baseline
trn, holdout = stratified_split(train_df, val_size=0.05, seed=SEED)

retr = PerLanguageRetriever().fit(trn)
retr_preds, retr_scores, _ = retr.predict(holdout)

baseline_overall  = compute_rouge(retr_preds, holdout[ANSWER_COL].tolist())
baseline_per_lang = compute_rouge_by_language(
    retr_preds, holdout[ANSWER_COL].tolist(), holdout[SUBSET_COL].tolist()
)

print("Retrieval Baseline — Overall ROUGE:")
for k, v in baseline_overall.items():
    print(f"  {k}: {v:.4f}")

print("\nPer-subset breakdown:")
display(baseline_per_lang.round(4))


## 4. Select Experiment Configuration

In [ ]:
# Auto-selects mt5_base (safe default) or mt5_large (≥15 GB VRAM)
cfg, config_path = select_config("configs")

print("\n" + cfg.summary())


In [ ]:
# ── Override for a specific experiment ───────────────────────────────────────
# Uncomment and edit to run a different experiment without editing the YAML:
#
# cfg.experiment_name     = "exp06_mt5base_lora_r32"
# cfg.lora_r              = 32
# cfg.lora_alpha          = 64
# cfg.learning_rate       = 3e-4
#
# Or load a specific YAML directly:
# cfg = load_config("configs/mt5_large.yaml")

print(f"Running: {cfg.experiment_name}")


## 5. Tokenisation

In [ ]:
tokenizer = load_tokenizer(cfg)

# Sanity check on a real Akan row (longest answers in dataset)
aka_row = train_df[train_df[SUBSET_COL] == "Aka_Gha"].iloc[0]
q_ids = tokenizer(aka_row["model_input"])["input_ids"]
a_ids = tokenizer(aka_row[ANSWER_COL])["input_ids"]
print(f"Akan input tokens : {len(q_ids)} (fits in {cfg.max_input_length}: {len(q_ids) <= cfg.max_input_length})")
print(f"Akan answer tokens: {len(a_ids)} (fits in {cfg.max_target_length}: {len(a_ids) <= cfg.max_target_length})")


In [ ]:
tok_train, tok_val = make_hf_datasets(
    train_df, val_df, tokenizer,
    max_input_len=cfg.max_input_length,
    max_target_len=cfg.max_target_length,
)
print("Tokenized train:", tok_train)
print("Tokenized val  :", tok_val)


## 6. Model Loading & Fine-tuning

In [ ]:
model = load_model(cfg, device=DEVICE)


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir                  = cfg.checkpoint_dir,
    num_train_epochs            = cfg.num_train_epochs,
    per_device_train_batch_size = cfg.per_device_train_batch_size,
    per_device_eval_batch_size  = cfg.per_device_eval_batch_size,
    gradient_accumulation_steps = cfg.gradient_accumulation_steps,
    learning_rate               = cfg.learning_rate,
    warmup_ratio                = cfg.warmup_ratio,
    weight_decay                = cfg.weight_decay,
    lr_scheduler_type           = cfg.lr_scheduler_type,
    fp16                        = (DEVICE == "cuda"),
    predict_with_generate       = True,
    generation_max_length       = cfg.max_target_length,
    evaluation_strategy         = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "rougeL",
    greater_is_better           = True,
    label_smoothing_factor      = cfg.label_smoothing_factor,
    logging_steps               = cfg.logging_steps,
    report_to                   = cfg.report_to,
    seed                        = cfg.seed,
    save_total_limit            = 2,
    dataloader_num_workers      = cfg.dataloader_num_workers,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model,
    label_pad_token_id=-100, pad_to_multiple_of=8,
)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = tok_train,
    eval_dataset    = tok_val,
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = make_compute_metrics(tokenizer),
    callbacks       = [EarlyStoppingCallback(
                           early_stopping_patience=cfg.early_stopping_patience)],
)

print(f"Experiment      : {cfg.experiment_name}")
print(f"Effective batch : {cfg.effective_batch_size()}")
print(f"Starting training...")
train_result = trainer.train()
print(f"Training complete — loss: {train_result.training_loss:.4f}")


In [ ]:
trainer.save_model(cfg.best_model_dir)
tokenizer.save_pretrained(cfg.best_model_dir)
print(f"Best model saved → {cfg.best_model_dir}")


## 7. Training Curves

In [ ]:
plot_training_curves(
    trainer.state.log_history,
    cfg.experiment_name,
    OUTPUT_DIR / "plots",
)


## 8. Evaluation — Per-Subset ROUGE

In [ ]:
# Evaluate on a stratified holdout from the validation set
holdout = holdout_for(cfg, holdout_fraction=0.10, seed=SEED)

overall, per_lang, val_preds = evaluate_model(
    trainer.model, tokenizer, holdout, cfg,
    batch_size=cfg.per_device_eval_batch_size,
    device=DEVICE,
)

print(f"\nOverall — ROUGE-1: {overall['rouge1']:.4f} | "
      f"ROUGE-2: {overall['rouge2']:.4f} | ROUGE-L: {overall['rougeL']:.4f}")
print("\nPer-subset breakdown:")
display(per_lang.round(4))


In [ ]:
# Figure 3: Per-subset ROUGE bar chart
plot_per_subset_rouge(per_lang, overall, cfg.experiment_name, OUTPUT_DIR / "plots")


In [ ]:
# Spot-check predictions
print("Sample predictions:")
for i in range(3):
    print(f"  [{holdout[SUBSET_COL].iloc[i]}]")
    print(f"  Q : {holdout[QUESTION_COL].iloc[i][:90]}")
    print(f"  GT: {holdout[ANSWER_COL].iloc[i][:90]}")
    print(f"  PR: {val_preds[i][:90]}")
    print()


## 9. Experiment Tracking

In [ ]:
# ── Experiment log — update val_r1, val_rL, lb after every run ───────────────
experiment_log = [
    {"id":"EXP-01","model":"mt5-small", "peft":"LoRA r=8,α=16",  "lr":"5e-4",
     "max_tgt":256,"prompt":"Plain Q",
     "val_r1":None,"val_rL":None,"lb":None,
     "finding":"Baseline — lower bound; establishes that retrieval-only ROUGE is the floor"},

    {"id":"EXP-02","model":"mt5-base",  "peft":"LoRA r=8,α=16",  "lr":"5e-4",
     "max_tgt":256,"prompt":"Plain Q",
     "val_r1":None,"val_rL":None,"lb":None,
     "finding":"Scale model — isolates model-size effect from prompt strategy"},

    {"id":"EXP-03","model":"mt5-base",  "peft":"LoRA r=16,α=32", "lr":"5e-4",
     "max_tgt":256,"prompt":"Subset prefix (Aka_Gha:...)",
     "val_r1":None,"val_rL":None,"lb":None,
     "finding":"Subset tag conditions model on language+country; 8 distinct generation targets"},

    {"id":"EXP-04","model":"mt5-base",  "peft":"LoRA r=16,α=32", "lr":"3e-4",
     "max_tgt":256,"prompt":"Subset prefix",
     "val_r1":None,"val_rL":None,"lb":None,
     "finding":"Lower LR — Amharic short answers (median 19w) overshoot at 5e-4"},

    {"id":"EXP-05","model":"mt5-base",  "peft":"LoRA r=16,α=32", "lr":"3e-4",
     "max_tgt":384,"prompt":"Subset prefix",
     "val_r1":None,"val_rL":None,"lb":None,
     "finding":"MAX_TARGET 256→384 — EDA: Akan p95=208w; 256 tok only covers 67.8% of all answers"},

    {"id":"EXP-06","model":"mt5-base",  "peft":"LoRA r=32,α=64", "lr":"3e-4",
     "max_tgt":384,"prompt":"Subset prefix",
     "val_r1":None,"val_rL":None,"lb":None,
     "finding":"Higher LoRA rank — more capacity for 8-way multilingual generation"},

    {"id":"EXP-07","model":"mt5-base",  "peft":"LoRA r=16,α=32", "lr":"3e-4",
     "max_tgt":384,"prompt":"Subset prefix",
     "val_r1":None,"val_rL":None,"lb":None,
     "finding":"Dedup: drop 467 intra-subset duplicates; retain 1,002 cross-subset dups"},

    {"id":"EXP-08","model":"mt5-base",  "peft":"Full fine-tune",  "lr":"1e-4",
     "max_tgt":384,"prompt":"Subset prefix",
     "val_r1":None,"val_rL":None,"lb":None,
     "finding":"All 580M params updated — compare parameter-efficiency vs LoRA"},

    {"id":"EXP-09","model":"mt5-large", "peft":"LoRA r=16,α=32", "lr":"3e-4",
     "max_tgt":384,"prompt":"Subset prefix",
     "val_r1":None,"val_rL":None,"lb":None,
     "finding":"Scale to mT5-large (1.2B params) — requires ≥14 GB VRAM"},

    {"id":"EXP-10","model":"mt5-base",  "peft":"LoRA r=16,α=32", "lr":"3e-4",
     "max_tgt":384,"prompt":"Subset prefix + label_smooth=0.1",
     "val_r1":None,"val_rL":None,"lb":None,
     "finding":"label_smoothing regularises low-resource subsets; num_beams=8 at inference"},
]

exp_df = pd.DataFrame(experiment_log)
exp_df[["id","model","peft","lr","max_tgt","prompt","val_r1","val_rL","lb"]]


In [ ]:
# ── Record current experiment result ─────────────────────────────────────────
# Run this cell after every training run to update the log.

current_exp_id = cfg.experiment_name.split("_")[0].upper()   # e.g. "EXP-03"
# current_lb = 0.000     # ← enter after uploading to Zindi

for row in experiment_log:
    if row["id"] == current_exp_id:
        row["val_r1"] = round(overall["rouge1"], 4)
        row["val_rL"] = round(overall["rougeL"], 4)
        # row["lb"]   = current_lb

exp_df = pd.DataFrame(experiment_log)
print(f"Logged {current_exp_id}: ROUGE-1={overall['rouge1']:.4f}, ROUGE-L={overall['rougeL']:.4f}")

reports_dir = Path(cfg.reports_dir)
reports_dir.mkdir(parents=True, exist_ok=True)
exp_df.to_csv(reports_dir / "experiments.csv", index=False)
print(f"Saved: {reports_dir / 'experiments.csv'}")


In [ ]:
# ── Load and display saved experiment log (all past runs) ─────────────────────
log_path = Path(cfg.reports_dir) / "experiments.csv"
if log_path.exists():
    display(pd.read_csv(log_path))
else:
    print("No saved experiment log yet — run a few experiments first.")


In [ ]:
# ── Leaderboard progression (run after ≥2 experiments have lb scores) ─────────
plot_leaderboard_progression(exp_df, OUTPUT_DIR / "plots")


## 10. Inference & Submission Generation

In [ ]:
print(f"Generating predictions for {len(test_df)} test records...")
test_preds = predict_dataframe(
    trainer.model, tokenizer, test_df, cfg,
    batch_size=cfg.per_device_eval_batch_size,
    device=DEVICE,
)
print(f"Generated {len(test_preds)} predictions.")

print("\nSpot check:")
for i in range(3):
    print(f"  [{test_df[SUBSET_COL].iloc[i]}]")
    print(f"  Q   : {test_df[QUESTION_COL].iloc[i][:90]}")
    print(f"  Pred: {test_preds[i][:90]}")
    print()


In [ ]:
submission = make_submission(
    ids=test_df[ID_COL].tolist(),
    predictions=test_preds,
    output_path=cfg.submission_path,
    sample_submission_path=str(DATA_DIR / "SampleSubmission.csv"),
)

print(f"\nSubmission shape : {submission.shape}")
print("Ready to upload to Zindi!")
display(submission.head(3))


## 11. Optional — LLM-as-a-Judge

In [ ]:
# Mirrors the competition's third evaluation axis using Gemini 1.5 Flash (free tier).
# Setup: pip install google-generativeai
#        Get a free key from https://ai.google.dev
#
# import google.generativeai as genai
# genai.configure(api_key="YOUR_GEMINI_API_KEY")
# judge = genai.GenerativeModel("gemini-1.5-flash")
#
# def llm_judge(question: str, reference: str, prediction: str) -> dict:
#     prompt = (
#         f"You are evaluating a health QA system for African languages.\n"
#         f"Question: {question}\n"
#         f"Reference answer: {reference}\n"
#         f"Predicted answer: {prediction}\n"
#         f"Score 1-5 for accuracy (medically correct), completeness, and fluency.\n"
#         f'Reply ONLY as valid JSON: {{\"accuracy\":N,\"completeness\":N,\"fluency\":N}}'
#     )
#     try:
#         r = judge.generate_content(prompt)
#         return json.loads(r.text.strip())
#     except Exception:
#         return {"accuracy": 0, "completeness": 0, "fluency": 0}
#
# holdout_sample = holdout.sample(min(50, len(holdout)), random_state=SEED)
# scores = [
#     llm_judge(row[QUESTION_COL], row[ANSWER_COL], val_preds[i])
#     for i, (_, row) in enumerate(tqdm(holdout_sample.iterrows(), total=len(holdout_sample)))
# ]
# judge_df = pd.DataFrame(scores)
# print(judge_df.describe().round(2))

print("LLM-as-a-Judge block ready — uncomment and add Gemini API key to run.")
